In [2]:
import time
from google.cloud import storage
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery

In [3]:
project_id = "auspicious-lead-382321"
bucket_name = "movie_data_lpez"
folder_name = "initial_load"
dataset_name = "movie_raw"
region = "us"

bq_client = bigquery.Client(project=project_id)

# Snacks

In [ ]:
%%bigquery
select * from movie_raw.amc_snack
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source,load_time
0,53,Gold Peak Brewed Tea - 16 fl oz,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,180.0,1.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00
1,54,Gold Peak Brewed Tea - 20 fl oz,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,220.0,2.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00
2,55,Gold Peak Brewed Tea - 30 fl oz,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,330.0,3.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00
3,56,Gold Peak Brewed Tea - 44 fl oz,15.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,40.0,480.0,4.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00
4,59,Apple Sauce,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,75.0,10.0,NaN,8.0,NaN,amc,2024-02-07 03:53:10.283111+00:00


In [ ]:
%%bigquery
select (select count(*) from movie_raw.amc_snack) as raw_snack_total_count,
  (select count(distinct id) from movie_raw.amc_snack) as raw_snack_distinct_count,
    (select count(*) - count(distinct id) from movie_raw.amc_snack) as duplicate_count

Query is running:   0%|          |

Downloading:   0%|          |

,raw_snack_total_count,raw_snack_distinct_count,duplicate_count
0,100,100,0


In [ ]:
%%bigquery
create or replace table movie_stg.Snack as
  select distinct * except(load_time), load_time
  from movie_raw.amc_snack

Query is running:   0%|          |

""


In [ ]:
%%bigquery
select * from movie_stg.Snack
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source,load_time
0,81,Mango Smoothie 16 fl oz,270.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN,72.0,NaN,68.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
1,59,Apple Sauce,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,75.0,10.0,NaN,8.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
2,79,Banana Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105.0,NaN,99.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
3,89,Strawberry Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102.0,NaN,93.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
4,84,Peach Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,0.0,104.0,NaN,101.0,NaN,amc,2024-02-07 03:53:10.283111+00:00


# Primary Key

In [4]:
%%bigquery
alter table movie_stg.Snack add primary key (id) not enforced

Executing query with job ID: 77db9459-e2f8-44a0-9d45-0cde93ef2b90
Query executing: 1.45s


ERROR:
 409 Already Exists: Constraint primary key; reason: duplicate, message: Already Exists: Constraint primary key

Location: US
Job ID: 77db9459-e2f8-44a0-9d45-0cde93ef2b90



In [ ]:
%%bigquery
select id, count(*) as duplicate_records
from movie_stg.Snack
group by id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,id,duplicate_records


# Actor

In [9]:
%%bigquery
select * from movie_raw.tmdb_actor
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,credit_id,person_id,movie_id,name,character,gender,credit_order,data_source,load_time
0,6271d29e29c6260067561c3e,122303,9829,Ben Sliney,Ben Sliney,0,53,tmdb,2024-02-14 01:27:38.706796+00:00
1,64eaa3bf06f98400ae4997d3,4236367,9829,Tobin Miller,Tobin Miller,0,54,tmdb,2024-02-14 01:27:38.706796+00:00
2,64eaa3e3c613ce00ac422294,4236368,9829,Rich Sullivan,Rich Sullivan,0,55,tmdb,2024-02-14 01:27:38.706796+00:00
3,64eaa3ef1feac100c46f1cab,4236369,9829,Tony Smith,Tony Smith,0,56,tmdb,2024-02-14 01:27:38.706796+00:00
4,64eaa434594c9400e26055e7,4236370,9829,Carol Bento,Herndon Staff,0,58,tmdb,2024-02-14 01:27:38.706796+00:00


In [10]:
%%bigquery
create or replace table movie_stg.Actor as
    SELECT *
    FROM movie_raw.tmdb_actor

Query is running:   0%|          |

""


In [12]:
%%bigquery
select credit_id, count(*) as duplicate_records
from movie_stg.Actor
group by credit_id
having count(*) > 1


Query is running:   0%|          |

Downloading: |          |

,credit_id,duplicate_records


# Primary Key

In [5]:
%%bigquery
alter table movie_stg.Actor add primary key (credit_id) not enforced

Query is running:   0%|          |

""


In [6]:
%%bigquery
select credit_id, count(*) as duplicate_records
from movie_stg.Actor
group by credit_id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,credit_id,duplicate_records


# Character

In [9]:
%%bigquery
select * from movie_raw.tmdb_character
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,person_id,movie_id,name,character,credit_order,data_source,load_time
0,122303,9829,Ben Sliney,Ben Sliney,53,tmdb,2024-02-09 21:29:30.927347+00:00
1,4236367,9829,Tobin Miller,Tobin Miller,54,tmdb,2024-02-09 21:29:30.927347+00:00
2,4236368,9829,Rich Sullivan,Rich Sullivan,55,tmdb,2024-02-09 21:29:30.927347+00:00
3,4236369,9829,Tony Smith,Tony Smith,56,tmdb,2024-02-09 21:29:30.927347+00:00
4,95195,9829,Michael Bofshever,John White,57,tmdb,2024-02-09 21:29:30.927347+00:00


In [13]:
%%bigquery
create or replace table movie_stg.Character as
    SELECT  *
    FROM movie_raw.tmdb_character

Query is running:   0%|          |

""


In [15]:
%%bigquery
select credit_id, count(*) as duplicate_records
from movie_stg.Character
group by credit_id
having count(*) > 1


Query is running:   0%|          |

Downloading: |          |

,credit_id,duplicate_records


# Primary Key

In [16]:
%%bigquery
alter table movie_stg.Character add primary key (credit_id) not enforced

Query is running:   0%|          |

""


In [18]:
%%bigquery
select credit_id, count(*) as duplicate_records
from movie_stg.Character
group by credit_id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,credit_id,duplicate_records


# Crew

In [14]:
%%bigquery
select * from movie_raw.tmdb_crew
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,credit_id,person_id,movie_id,name,gender,department,known_for_department,job,data_source,load_time
0,6067975ff056d500295f8f79,3035237,805051,Erhard Dietl,0,Writing,Writing,Book,tmdb,2024-02-14 00:20:25.380059+00:00
1,5bfdfda4925141165403c0f4,54039,8197,John Berendt,0,Writing,Writing,Book,tmdb,2024-02-14 00:20:25.380059+00:00
2,59234501c3a3680491000a37,1821750,363992,Diana Henriques,0,Writing,Writing,Book,tmdb,2024-02-14 00:20:25.380059+00:00
3,5c43627b9251412bf6dc3528,2221568,63193,Lisa Wysocky,0,Writing,Writing,Book,tmdb,2024-02-14 00:20:25.380059+00:00
4,5c43629fc3a3681a0322d041,2221569,63193,Brad Cohen,0,Writing,Writing,Book,tmdb,2024-02-14 00:20:25.380059+00:00


In [19]:
%%bigquery
create or replace table movie_stg.Crew as
    SELECT *
    FROM movie_raw.tmdb_crew

Query is running:   0%|          |

""


In [21]:
%%bigquery
select * from movie_stg.Crew
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,credit_id,person_id,movie_id,name,gender,department,known_for_department,job,data_source,load_time
0,572db4869251410d3000178b,1616371,297806,Colin Escott,0,Writing,Writing,Book,tmdb,2024-02-14 01:27:22.319767+00:00
1,61dfca8ce24b93006689254e,1274243,50321,Berkeley Breathed,0,Writing,Writing,Book,tmdb,2024-02-14 01:27:22.319767+00:00
2,5fb4462dd55e4d003ed6940e,2040822,523781,Julia Walton,0,Writing,Writing,Book,tmdb,2024-02-14 01:27:22.319767+00:00
3,6483830de375c0011c7f4dd2,4104695,1138248,Gerard Jofra,0,Writing,Writing,Book,tmdb,2024-02-14 01:27:22.319767+00:00
4,5fcad31e43999b003e1816cf,2360093,614911,Lily Brooks-Dalton,0,Writing,Writing,Book,tmdb,2024-02-14 01:27:22.319767+00:00


# Primary Key

In [22]:
%%bigquery
alter table movie_stg.Crew add primary key (credit_id) not enforced

Query is running:   0%|          |

""


In [23]:
%%bigquery
select credit_id, count(*) as duplicate_records
from movie_stg.Crew
group by credit_id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,credit_id,duplicate_records


# Genre

In [40]:
%%bigquery
select * from movie_raw.tmdb_genre
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,genre_id,name,data_source,load_time
0,28,Action,tmdb,2024-02-07 03:41:19.340497+00:00
1,12,Adventure,tmdb,2024-02-07 03:41:19.340497+00:00
2,16,Animation,tmdb,2024-02-07 03:41:19.340497+00:00
3,35,Comedy,tmdb,2024-02-07 03:41:19.340497+00:00
4,80,Crime,tmdb,2024-02-07 03:41:19.340497+00:00


In [42]:
%%bigquery
create or replace table movie_stg.Genre as
  select distinct *
  from movie_raw.tmdb_genre

Query is running:   0%|          |

""


In [7]:
%%bigquery
select genre_id, count(*) as duplicate_records
from movie_stg.Genre
group by genre_id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,genre_id,duplicate_records


In [43]:
%%bigquery
select * from movie_stg.Genre
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,genre_id,name,data_source,load_time
0,16,Animation,tmdb,2024-02-07 03:41:19.340497+00:00
1,18,Drama,tmdb,2024-02-07 03:41:19.340497+00:00
2,9648,Mystery,tmdb,2024-02-07 03:41:19.340497+00:00
3,53,Thriller,tmdb,2024-02-07 03:41:19.340497+00:00
4,10770,TV Movie,tmdb,2024-02-07 03:41:19.340497+00:00


# Primary Key

In [8]:
%%bigquery
alter table movie_stg.Genre add primary key (genre_id) not enforced

Query is running:   0%|          |

""


# Movie_Genre

In [45]:
%%bigquery
select * from movie_raw.tmdb_movie_genre
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,genre_id,data_source,load_time
0,940551,12,tmdb,2024-02-07 03:41:55.938198+00:00
1,940551,12,tmdb,2024-02-07 03:41:55.938198+00:00
2,940551,12,tmdb,2024-02-07 03:41:55.938198+00:00
3,940551,12,tmdb,2024-02-07 03:41:55.938198+00:00
4,940551,12,tmdb,2024-02-07 03:41:55.938198+00:00


In [47]:
%%bigquery
select
  (select count(*) from movie_raw.tmdb_movie_genre) as raw_movie_genre_total_count,
  (select count(distinct concat(movie_id, '-', genre_id)) from movie_raw.tmdb_movie_genre) as raw_movie_genre_distinct_count

Query is running:   0%|          |

Downloading:   0%|          |

,raw_movie_genre_total_count,raw_movie_genre_distinct_count
0,29000,58


In [51]:
%%bigquery
create or replace table movie_stg.Movie_Genre as
  select distinct concat(movie_id, '-', genre_id) as movie_genre_id, *
  from movie_raw.tmdb_movie_genre

Query is running:   0%|          |

""


In [52]:
%%bigquery
select * from movie_stg.Movie_Genre
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_genre_id,movie_id,genre_id,data_source,load_time
0,753342-10752,753342,10752,tmdb,2024-02-07 03:41:55.938198+00:00
1,918692-10752,918692,10752,tmdb,2024-02-07 03:41:55.938198+00:00
2,508883-12,508883,12,tmdb,2024-02-07 03:41:55.938198+00:00
3,572802-12,572802,12,tmdb,2024-02-07 03:41:55.938198+00:00
4,346698-12,346698,12,tmdb,2024-02-07 03:41:55.938198+00:00


In [54]:
%%bigquery
select (select count(*) from movie_stg.Movie_Genre) as stg_movie_genre_records,
  (select count(*) from movie_raw.tmdb_movie_genre) as raw_movie_genre_records

Query is running:   0%|          |

Downloading:   0%|          |

,stg_movie_genre_records,raw_movie_genre_records
0,58,29000


**Primary Key**

In [55]:
%%bigquery
alter table movie_stg.Movie_Genre add primary key (movie_genre_id) not enforced

Query is running:   0%|          |

""


In [56]:
%%bigquery
select movie_genre_id, count(*) as duplicate_records
from movie_stg.Movie_Genre
group by movie_genre_id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,movie_genre_id,duplicate_records
